In [ ]:
!pip install -Uqqq pip --progress-bar off
!pip install -qqq openai==1.82.0 --progress-bar off
!pip install -qqq python-dotenv==1.1.0 --progress-bar off
!pip install -qqq pydantic==2.11.7 --progress-bar off

In [ ]:
import json
import os
import textwrap
from typing import Literal

from dotenv import load_dotenv
from IPython.display import Markdown, display
from openai import OpenAI
from pydantic import BaseModel

load_dotenv()

MODEL_NAME = "MiniMax-M2.7"

client = OpenAI(
    api_key=os.environ["MINIMAX_API_KEY"],
    base_url="https://api.minimax.io/v1",
)


def format_response(response):
    response_txt = response.choices[0].message.content
    text = ""
    for chunk in response_txt.split("\n"):
        text += "\n"
        if not chunk:
            continue
        text += (
            "\n".join(textwrap.wrap(chunk, 100, break_long_words=False))
        ).strip()
    return text.strip()

## Text Generation

[MiniMax M2.7](https://www.minimax.io/) is a large language model with a 204K context window.
It provides an OpenAI-compatible API, so you can use the standard `openai` Python SDK.

In [ ]:
%%time

messages = [
    {"role": "user", "content": "Explain what is Deep Learning in one sentence"},
]

response = client.chat.completions.create(
    model=MODEL_NAME, messages=messages, temperature=0.1
)

In [ ]:
print(textwrap.fill(response.choices[0].message.content, 120))

In [ ]:
usage = response.usage
print(
    f"""
Tokens Used

Prompt:     {usage.prompt_tokens}
Completion: {usage.completion_tokens}
Total:      {usage.total_tokens}
"""
)

## System Prompt

In [ ]:
%%time

messages = [
    {
        "role": "system",
        "content": "You are a helpful AI coding assistant. Always provide clear, concise answers with code examples.",
    },
    {
        "role": "user",
        "content": "How do I read a CSV file in Python using pandas?",
    },
]

response = client.chat.completions.create(
    model=MODEL_NAME, messages=messages, temperature=0.1
)

In [ ]:
display(Markdown(format_response(response)))

## Streaming

In [ ]:
messages = [
    {"role": "user", "content": "Write a short poem about machine learning"},
]

completion = client.chat.completions.create(
    model=MODEL_NAME, messages=messages, temperature=0.7, stream=True
)

for chunk in completion:
    content = chunk.choices[0].delta.content
    if content is not None:
        print(content, end="")

## JSON Response

In [ ]:
%%time

messages = [
    {
        "role": "user",
        "content": "List the top 3 programming languages for AI development. Return a JSON object with a 'languages' key containing a list of objects with 'name', 'reason', and 'popularity_rank' fields.",
    },
]

response = client.chat.completions.create(
    model=MODEL_NAME,
    messages=messages,
    response_format={"type": "json_object"},
    temperature=0.1,
)

In [ ]:
result = json.loads(response.choices[0].message.content)
print(json.dumps(result, indent=2))

## Structured Output with Pydantic

In [ ]:
class SentimentClassification(BaseModel):
    sentiment: Literal["negative", "neutral", "positive"]
    reasoning: str


PROMPT = """
Classify the text sentiment into one of negative, neutral or positive.
Give your reasoning in the `reasoning` field.

Text:
```
I am very happy to say that AI has taken my job, for good!
```

Respond with a JSON object matching this schema:
{"sentiment": "negative|neutral|positive", "reasoning": "your reasoning"}
"""

messages = [{"role": "user", "content": PROMPT}]

response = client.chat.completions.create(
    model=MODEL_NAME,
    messages=messages,
    response_format={"type": "json_object"},
    temperature=0.1,
)

In [ ]:
json_content = response.choices[0].message.content
result = SentimentClassification.model_validate_json(json_content)
print(f"Sentiment: {result.sentiment}")
print(f"Reasoning: {result.reasoning}")

## Simulating a Chat

In [ ]:
%%time

messages = [
    {
        "role": "system",
        "content": "You are a helpful AI tutor specializing in machine learning.",
    },
    {"role": "user", "content": "What is the difference between supervised and unsupervised learning?"},
    {
        "role": "assistant",
        "content": "Supervised learning uses labeled data to train models, while unsupervised learning finds patterns in unlabeled data.",
    },
    {
        "role": "user",
        "content": "Give me an example of each.",
    },
]

response = client.chat.completions.create(
    model=MODEL_NAME, messages=messages, temperature=0.1
)

In [ ]:
display(Markdown(format_response(response)))

## Using Tools (Function Calling)

In [ ]:
def get_weather(city: str, unit: str = "celsius") -> str:
    """Get the current weather for a city.

    Parameters
    ----------
    city : str
        The city name
    unit : str
        Temperature unit (celsius or fahrenheit)

    Returns
    -------
    str
        Weather information as JSON string
    """
    weather_data = {
        "San Francisco": {"temp": 18, "condition": "Foggy"},
        "New York": {"temp": 25, "condition": "Sunny"},
        "London": {"temp": 15, "condition": "Rainy"},
        "Tokyo": {"temp": 28, "condition": "Humid"},
    }
    data = weather_data.get(city, {"temp": 20, "condition": "Unknown"})
    if unit == "fahrenheit":
        data["temp"] = round(data["temp"] * 9 / 5 + 32)
    data["unit"] = unit
    data["city"] = city
    return json.dumps(data)

In [ ]:
tools = [
    {
        "type": "function",
        "function": {
            "name": "get_weather",
            "description": "Get the current weather for a city",
            "parameters": {
                "type": "object",
                "properties": {
                    "city": {
                        "type": "string",
                        "description": "The city name",
                    },
                    "unit": {
                        "type": "string",
                        "enum": ["celsius", "fahrenheit"],
                        "description": "Temperature unit",
                    },
                },
                "required": ["city"],
            },
        },
    }
]

messages = [
    {"role": "user", "content": "What's the weather like in Tokyo?"}
]

response = client.chat.completions.create(
    model=MODEL_NAME,
    messages=messages,
    tools=tools,
    tool_choice="auto",
    temperature=0.1,
)

In [ ]:
response_message = response.choices[0].message
tool_calls = response_message.tool_calls
print(f"Tool calls: {tool_calls}")

In [ ]:
available_functions = {"get_weather": get_weather}

messages.append(response_message)

for tool_call in tool_calls:
    function_name = tool_call.function.name
    function_to_call = available_functions[function_name]
    function_args = json.loads(tool_call.function.arguments)
    function_response = function_to_call(**function_args)
    messages.append(
        {
            "tool_call_id": tool_call.id,
            "role": "tool",
            "name": function_name,
            "content": function_response,
        }
    )

In [ ]:
%%time

final_response = client.chat.completions.create(
    model=MODEL_NAME, messages=messages, temperature=0.1
)

print(final_response.choices[0].message.content)

## Text Summarization

In [ ]:
%%time

article = """
Artificial intelligence has made remarkable strides in recent years, particularly in the field of
natural language processing. Large language models, trained on vast amounts of text data, have
demonstrated the ability to generate human-like text, translate languages, write different kinds
of creative content, and answer questions in an informative way. These models use transformer
architectures that process text through attention mechanisms, allowing them to capture long-range
dependencies in language. The implications for industries ranging from healthcare to education
are profound, as these models can assist with everything from medical diagnosis to personalized
tutoring. However, challenges remain around hallucination, bias, and the environmental cost of
training increasingly large models.
"""

messages = [
    {
        "role": "user",
        "content": f"Summarize the following text in 2-3 sentences:\n\n{article}",
    },
]

response = client.chat.completions.create(
    model=MODEL_NAME, messages=messages, temperature=0.1
)

print(textwrap.fill(response.choices[0].message.content, 120))

## Data Labelling

In [ ]:
CLASSIFY_TEXT_PROMPT = """
Your task is to analyze the following text and classify it based on multiple criteria.
Provide your analysis as a JSON object. Use only the specified categories:

1. Target audience:
   ['General public', 'Professionals', 'Academics', 'Students']

2. Tone:
   ['Neutral', 'Positive', 'Negative', 'Formal', 'Informal', 'Humorous']

3. Complexity level:
   ['Elementary', 'Intermediate', 'Advanced', 'Technical']

4. Main topic:
   ['Technology', 'Science', 'Health', 'Business', 'Education', 'Entertainment']

For each classification, choose the most appropriate category.

<text>
{text}
</text>

Respond with JSON: {{"target_audience": "...", "tone": "...", "complexity": "...", "topic": "..."}}
"""

texts = [
    "The new iPhone 16 features an improved camera system and A18 chip.",
    "Deep reinforcement learning combines neural networks with reward-based training.",
    "Five easy stretches you can do at your desk to reduce back pain!",
]

results = []
for text in texts:
    response = client.chat.completions.create(
        model=MODEL_NAME,
        messages=[{"role": "user", "content": CLASSIFY_TEXT_PROMPT.format(text=text)}],
        response_format={"type": "json_object"},
        temperature=0.1,
    )
    result = json.loads(response.choices[0].message.content)
    result["text"] = text[:60]
    results.append(result)

for r in results:
    print(json.dumps(r, indent=2))
    print()

## Using the Highspeed Model

MiniMax also offers `MiniMax-M2.7-highspeed` for faster inference at a lower cost.

In [ ]:
%%time

response = client.chat.completions.create(
    model="MiniMax-M2.7-highspeed",
    messages=[{"role": "user", "content": "Explain what is Deep Learning in one sentence"}],
    temperature=0.1,
)

print(textwrap.fill(response.choices[0].message.content, 120))